# Other organisms — deep dive (mouse + pig)

This notebook is a marketing appendix that strengthens the claim that IDTrack’s core semantics generalize beyond human.

## Rationale

The main manuscript focuses on human, but a reviewer may ask whether the approach is human-specific.
This notebook provides *bounded, cache-first* evidence that for other Ensembl organisms:

- The graph snapshot can be reused from the shared local repository
- Outcome semantics (1→0 / 1→1 / 1→n) behave as expected
- The time axis (target release sweep) produces stable, interpretable trends

Scope: within-species conversions only (no ortholog mapping).

## Outputs

- `_outputs/_publication/figures/fig_other_organisms_deep_dive.pdf` (Ensembl backbone time travel)
- `_outputs/_publication/figures/fig_other_organisms_deep_dive_uniprot.pdf` (external matching across time)
- `_outputs/_publication/tables/other_organisms_portability_summary.csv`
- Cached summaries under `idtrack/docs/_notebooks/idtrack_cache/experiments/other_organisms_deep_dive/`


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:  # noqa: S110
    sns = None

import os
import sys

# Add experiments/src to sys.path (layout-aware; works on Slurm and locally)
REPO_ROOT = Path(os.environ.get('REPO_ROOT', Path.cwd())).expanduser().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (
    (REPO_ROOT / 'idtrack').is_dir()
    and ((REPO_ROOT / 'reproducibility').is_dir() or (REPO_ROOT / 'idtrack' / 'reproducibility').is_dir())
):
    REPO_ROOT = REPO_ROOT.parent

REPRO_ROOT = REPO_ROOT / 'reproducibility' if (REPO_ROOT / 'reproducibility').is_dir() else REPO_ROOT / 'idtrack' / 'reproducibility'
EXPERIMENTS_SRC = REPRO_ROOT / 'experiments' / 'src'
if str(EXPERIMENTS_SRC) not in sys.path:
    sys.path.insert(0, str(EXPERIMENTS_SRC))

from experiments_utils import (  # noqa: E402
    MANUSCRIPT_COLORS,
    notebook_context,
    read_pickle,
    save_figure,
    write_pickle,
)

from idtrack_results import summarize_binned_conversion  # noqa: E402

ctx = notebook_context('other_organisms_deep_dive', start=REPO_ROOT)
plt.rcParams.update({'savefig.dpi': 300})

IDTRACK_LOCAL_REPO = ctx.idtrack_local_repo
CACHE_DIR = ctx.experiment_cache
MANUSCRIPT_FIGURES = ctx.manuscript_figures

print('IDTRACK_LOCAL_REPO:', IDTRACK_LOCAL_REPO)
print('CACHE_DIR:', CACHE_DIR)


In [ ]:
# -------------------- Configuration --------------------

ORGANISMS = [
    {'alias': 'mouse', 'prefix': 'ENSMUSG'},
    {'alias': 'pig', 'prefix': 'ENSSSCG'},
]

SNAPSHOT_RELEASE = 114  # must be <= what you have cached
TARGET_RELEASES = list(range(100, 115, 3))  # sweep for the time-axis plot

N_SAMPLE_IDS = 300
STRATEGY = 'all'

# Evaluate both backbone-only time travel and an external-matching target.
# HGNC is human-specific; UniProt is a portable external namespace for marketing portability.
FINAL_DATABASES = [
    None,
    'UniProtKB/Swiss-Prot',
]

RESULTS_PKL = CACHE_DIR / (
    f"other_organisms_deep_dive_snapshot{SNAPSHOT_RELEASE}_to{TARGET_RELEASES[0]}-{TARGET_RELEASES[-1]}_"
    f"n{N_SAMPLE_IDS}_strategy{STRATEGY}.pickle"
)

print('RESULTS_PKL:', RESULTS_PKL)


In [ ]:
# -------------------- Compute (cache-first) --------------------

import idtrack

needs_recompute = True
if RESULTS_PKL.exists():
    payload = read_pickle(RESULTS_PKL)
    print('Loaded:', RESULTS_PKL)
    try:
        rows0 = payload.get('rows', []) if isinstance(payload, dict) else []
        if rows0 and {'final_database', '1_to_1_tdm', '1_to_n_tdm', '1_to_1_atm', '1_to_n_atm'}.issubset(set(rows0[0])):
            needs_recompute = False
    except Exception:
        needs_recompute = True

if needs_recompute:
    rng = np.random.default_rng(0)

    api = idtrack.API(local_repository=str(IDTRACK_LOCAL_REPO))
    api.configure_logger()

    def reservoir_sample(nodes, prefix: str, k: int) -> list[str]:
        sample: list[str] = []
        n_seen = 0
        for node in nodes:
            if not isinstance(node, str) or not node.startswith(prefix):
                continue
            n_seen += 1
            if len(sample) < k:
                sample.append(node)
                continue
            j = int(rng.integers(0, n_seen))
            if j < k:
                sample[j] = node
        return sample

    rows = []
    for org in ORGANISMS:
        organism_alias = org['alias']
        prefix = org['prefix']

        organism, latest = api.resolve_organism(organism_alias)
        snapshot = min(int(SNAPSHOT_RELEASE), int(latest))
        api.build_graph(organism_name=organism, snapshot_release=snapshot, calculate_caches=True)

        ids = reservoir_sample(api.track.graph.nodes, prefix=prefix, k=N_SAMPLE_IDS)
        if not ids:
            raise RuntimeError(f'Could not sample IDs for {organism_alias} with prefix {prefix!r}.')

        for final_db in FINAL_DATABASES:
            for to_release in TARGET_RELEASES:
                matchings = api.convert_identifier_multiple(
                    ids.copy(),
                    from_release=int(snapshot),
                    to_release=int(to_release),
                    final_database=final_db,
                    strategy=STRATEGY,
                )
                bins = api.classify_multiple_conversion(matchings)
                stats = summarize_binned_conversion(bins)

                changed = int(stats['changed_1_to_1']) + int(stats['changed_1_to_n'])
                total = int(stats['total'])

                rows.append(
                    {
                        'organism': organism_alias,
                        'to_release': int(to_release),
                        'final_database': final_db if final_db is not None else 'Ensembl gene',
                        'n': total,
                        '1_to_0': int(stats['1_to_0']),
                        '1_to_1_tdm': int(stats['1_to_1_tdm']),
                        '1_to_1_atm': int(stats['1_to_1_atm']),
                        '1_to_n_tdm': int(stats['1_to_n_tdm']),
                        '1_to_n_atm': int(stats['1_to_n_atm']),
                        'changed_1_to_1': int(stats['changed_1_to_1']),
                        'changed_1_to_n': int(stats['changed_1_to_n']),
                        'frac_changed': float(changed / total) if total else float('nan'),
                    }
                )

    payload = {
        'params': {
            'snapshot_release': int(SNAPSHOT_RELEASE),
            'target_releases': [int(r) for r in TARGET_RELEASES],
            'n_sample_ids': int(N_SAMPLE_IDS),
            'strategy': str(STRATEGY),
            'final_databases': [db if db is not None else None for db in FINAL_DATABASES],
        },
        'rows': rows,
    }
    write_pickle(payload, RESULTS_PKL)
    print('Saved:', RESULTS_PKL)

df = pd.DataFrame(payload['rows'])
df.head()


In [ ]:
# -------------------- Plot: multi-panel manuscript-ready figure --------------------

df = pd.DataFrame(payload['rows'])

def _with_totals(d: pd.DataFrame) -> pd.DataFrame:
    d = d.copy()
    d['1_to_1_total'] = d['1_to_1_tdm'] + d['1_to_1_atm']
    d['1_to_n_total'] = d['1_to_n_tdm'] + d['1_to_n_atm']
    d['atm_total'] = d['1_to_1_atm'] + d['1_to_n_atm']
    d['tdm_total'] = d['1_to_1_tdm'] + d['1_to_n_tdm']
    d['changed_total'] = d['changed_1_to_1'] + d['changed_1_to_n']
    return d

df = _with_totals(df)

def _plot_time_axis(df_sub: pd.DataFrame, *, title_prefix: str, out_name: str, external_mode: bool) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(12.5, 8.4), constrained_layout=True)
    ax0, ax1, ax2, ax3 = axes.ravel()

    mid_release = int(TARGET_RELEASES[len(TARGET_RELEASES) // 2])
    sub = df_sub[df_sub['to_release'] == mid_release].set_index('organism')

    n = sub['n'].replace(0, np.nan)
    if external_mode:
        frac = pd.DataFrame(
            {
                '1→0': sub['1_to_0'] / n,
                '1→1 TDM': sub['1_to_1_tdm'] / n,
                '1→1 ATM': sub['1_to_1_atm'] / n,
                '1→n TDM': sub['1_to_n_tdm'] / n,
                '1→n ATM': sub['1_to_n_atm'] / n,
            }
        )
        colors = [
            MANUSCRIPT_COLORS['1→0'],
            MANUSCRIPT_COLORS['1→1 TDM'],
            MANUSCRIPT_COLORS['1→1 ATM'],
            MANUSCRIPT_COLORS['1→n TDM'],
            MANUSCRIPT_COLORS['1→n ATM'],
        ]
        legend_title = 'Outcome (external matching)'
    else:
        frac = pd.DataFrame(
            {
                '1→0': sub['1_to_0'] / n,
                '1→1': sub['1_to_1_total'] / n,
                '1→n': sub['1_to_n_total'] / n,
            }
        )
        colors = [MANUSCRIPT_COLORS['1→0'], MANUSCRIPT_COLORS['1→1'], MANUSCRIPT_COLORS['1→n']]
        legend_title = 'Outcome (Ensembl backbone)'

    frac.plot(kind='bar', stacked=True, ax=ax0, color=colors)
    ax0.set_ylim(0, 1)
    ax0.set_ylabel('Fraction of queries')
    ax0.set_title(f"A) Outcome profile (to_release={mid_release})")
    ax0.legend(loc='upper right', frameon=True, title=legend_title)


    if external_mode:
        # Panel B: true external match fraction
        for org in [o['alias'] for o in ORGANISMS]:
            d = df_sub[df_sub['organism'] == org].sort_values('to_release')
            ax1.plot(d['to_release'], d['tdm_total'] / d['n'].replace(0, np.nan), '-o', label=org, lw=1.3, ms=3)
        ax1.set_ylim(0, 1)
        ax1.set_xlabel('Target Ensembl release')
        ax1.set_ylabel('Fraction')
        ax1.set_title('B) Any external match (TDM)')
        ax1.legend(frameon=True)

        # Panel C: fallback-to-Ensembl rate (ATM)
        for org in [o['alias'] for o in ORGANISMS]:
            d = df_sub[df_sub['organism'] == org].sort_values('to_release')
            ax2.plot(d['to_release'], d['atm_total'] / d['n'].replace(0, np.nan), '-o', label=org, lw=1.3, ms=3)
        ax2.set_ylim(0, 1)
        ax2.set_xlabel('Target Ensembl release')
        ax2.set_ylabel('Fraction')
        ax2.set_title('C) External missing (ATM fallback)')
        ax2.legend(frameon=True)
    else:
        for org in [o['alias'] for o in ORGANISMS]:
            d = df_sub[df_sub['organism'] == org].sort_values('to_release')
            ax1.plot(d['to_release'], d['1_to_1_total'] / d['n'].replace(0, np.nan), '-o', label=org, lw=1.3, ms=3)
            ax2.plot(d['to_release'], d['1_to_n_total'] / d['n'].replace(0, np.nan), '-o', label=org, lw=1.3, ms=3)
        ax1.set_ylim(0, 1)
        ax2.set_ylim(0, 1)
        ax1.set_xlabel('Target Ensembl release')
        ax2.set_xlabel('Target Ensembl release')
        ax1.set_ylabel('Fraction')
        ax2.set_ylabel('Fraction')
        ax1.set_title('B) Stability (1→1 fraction)')
        ax2.set_title('C) Ambiguity (1→n fraction)')
        ax1.legend(frameon=True)
        ax2.legend(frameon=True)

    # Panel D: drift signal
    for org in [o['alias'] for o in ORGANISMS]:
        d = df_sub[df_sub['organism'] == org].sort_values('to_release')
        ax3.plot(d['to_release'], d['changed_total'] / d['n'].replace(0, np.nan), '-o', label=org, lw=1.3, ms=3)
    ax3.set_ylim(0, 1)
    ax3.set_xlabel('Target Ensembl release')
    ax3.set_ylabel('Fraction')
    ax3.set_title('D) Drift signal (changed-only)')
    ax3.legend(frameon=True)

    fig.suptitle(title_prefix)
    written = save_figure(fig, out_name, ctx, formats=('pdf',))
    print('Saved:', written['pdf'])

df_backbone = df[df['final_database'] == 'Ensembl gene'].copy()
df_uniprot = df[df['final_database'] == 'UniProtKB/Swiss-Prot'].copy()

_plot_time_axis(df_backbone, title_prefix='Other organisms: Ensembl-backbone time travel', out_name='fig_other_organisms_deep_dive.pdf', external_mode=False)
if not df_uniprot.empty:
    _plot_time_axis(df_uniprot, title_prefix='Other organisms: external matching (UniProt) across time', out_name='fig_other_organisms_deep_dive_uniprot.pdf', external_mode=True)


## Marketing extension: portability summary table

A concise way to report “IDTrack is not human-only” is a small table that summarizes, per organism and target definition:

- mean 1→0 (coverage loss)
- mean 1→n (ambiguity)
- mean TDM total (stable success)

computed across the swept `to_release` values.


In [ ]:
from experiments_utils import atomic_write_dataframe_csv  # noqa: E402

if df.empty:
    print('No rows; skipping portability summary.')
else:
    summ = (
        df.groupby(['organism', 'final_database'], as_index=False)[
            ['frac_1_to_0', 'frac_1_to_n_total', 'frac_tdm_total', 'frac_atm_total', 'frac_changed_any']
        ]
        .mean(numeric_only=True)
        .sort_values(['organism', 'final_database'])
    )
    out_csv = ctx.manuscript_tables / 'other_organisms_portability_summary.csv'
    atomic_write_dataframe_csv(summ, out_csv, index=False)
    atomic_write_dataframe_csv(summ, ctx.experiment_outputs / 'tables' / out_csv.name, index=False)
    print('Wrote:', out_csv)
    summ
